In [ ]:
%pip install -qU pandas tqdm

from tqdm import tqdm

# Utils


In [ ]:
%pip install -q newspaper3k lxml

from newspaper import Article


def fetch_one_article(url: str) -> str | None:
    try:
        article = Article(url)
        article.download()
        article.parse()
        return article.text
    except Exception as e:
        print(f"Error fetching {url}: {str(e)}")
        return None

In [ ]:
from urllib.parse import urlparse


def get_domain(url):
    return urlparse(url).netloc.split("www.")[-1]

### Load not fetchable domains

Domains that we know can't be fetched using newspaper3k because of restrictions


In [ ]:
NOT_FETCHABLE_DOMAINS = open("not_fechable_domains.txt").read().splitlines()

NOT_FETCHABLE_DOMAINS[:3]

# Update MongoDB with article contents


In [ ]:
from dotenv import load_dotenv
from pymongo import MongoClient
import os

load_dotenv()

client = MongoClient(os.getenv("MONGODB_URI"))

db = client["blogdb"]

collection = db["ai_news"]

### Add a domain field


In [ ]:
from pymongo import UpdateOne

# Batch size for processing
BATCH_SIZE = 1000

# Find all documents without a domain field
docs_without_domain = collection.find(
    {"domain": {"$exists": False}}, {"_id": 1, "url": 1}
)

# Count total documents to update for the progress bar
total_docs = collection.count_documents({"domain": {"$exists": False}})

updated_count = 0
bulk_operations = []

# Process documents in batches
for doc in tqdm(docs_without_domain, total=total_docs, desc="Processing documents"):
    if "url" in doc:
        domain = get_domain(doc["url"])
        bulk_operations.append(
            UpdateOne({"_id": doc["_id"]}, {"$set": {"domain": domain}})
        )

    # If we've reached the batch size, execute the bulk operation
    if len(bulk_operations) >= BATCH_SIZE:
        result = collection.bulk_write(bulk_operations)
        updated_count += result.modified_count
        bulk_operations = []

# Execute any remaining operations
if bulk_operations:
    result = collection.bulk_write(bulk_operations)
    updated_count += result.modified_count

print(f"Added domain to {updated_count} documents")

# Verify that all documents now have a domain
remaining_docs = collection.count_documents({"domain": {"$exists": False}})
assert (
    remaining_docs == 0
), f"There are still {remaining_docs} documents without a domain"

In [ ]:
import time
import random
from tqdm import tqdm
from collections import defaultdict


def process_articles():
    # Query for documents without page_content and with fetchable domains
    query = {
        "page_content": {"$exists": False},
        "domain": {"$nin": NOT_FETCHABLE_DOMAINS},
    }

    projection = {"_id": 1, "url": 1}

    # Get total count for progress bar
    total_docs = collection.count_documents(query)

    previous_domain = None
    domain_errors = defaultdict(int)
    ignored_domains = set()

    for doc in tqdm(
        collection.find(query, projection), total=total_docs, desc="Processing articles"
    ):
        url = doc["url"]
        current_domain = get_domain(url)

        # Skip if domain is in ignored list
        if current_domain in ignored_domains:
            print(f"Skipping {url} (domain ignored due to multiple failures)")
            continue

        # If the domain is the same as the previous request, add a delay
        if current_domain == previous_domain:
            time.sleep(random.uniform(0.5, 2))

        content = fetch_one_article(url)

        if content:
            result = collection.update_one(
                {"_id": doc["_id"]}, {"$set": {"page_content": content}}
            )
            print(f"Updated {url}")
            # Reset error count for successful fetch
            domain_errors[current_domain] = 0
        else:
            print(f"No content fetched for {url}")
            domain_errors[current_domain] += 1

            # Check if domain should be ignored
            if domain_errors[current_domain] >= 3:
                ignored_domains.add(current_domain)
                print(f"Domain {current_domain} added to ignored list after 3 failures")

        previous_domain = current_domain

    print(f"Domains ignored due to multiple failures: {ignored_domains}")
    return ignored_domains


# Main execution
try:
    newly_ignored_domains = process_articles()

    # Optionally, update NOT_FETCHABLE_DOMAINS with newly ignored domains
    NOT_FETCHABLE_DOMAINS.extend(newly_ignored_domains)
    print(f"Updated NOT_FETCHABLE_DOMAINS: {NOT_FETCHABLE_DOMAINS}")

except KeyboardInterrupt:
    print("Process interrupted by user. Progress saved.")

# Update a CSV file


In [ ]:
%pip install -q pandas

import pandas as pd

### Load data from CSV


In [ ]:
FILE = r"data\blogdb.ai_news_2024_07_02_no_embedding.csv"

df = pd.read_csv(FILE)

df.head()

### Add `domain` field


In [ ]:
df["domain"] = df["url"].apply(get_domain)

df.head()

# Find not featchable domains


In [ ]:
domain_popularity = df["domain"].value_counts()

domain_popularity.head(10)

In [ ]:
one_article_per_domain = (
    df.sort_values("date", ascending=False)
    .drop_duplicates(subset="domain")
    .assign(domain_popularity=lambda x: x["domain"].map(domain_popularity))
    .sort_values("domain_popularity", ascending=False)
    .reset_index(drop=True)
)

one_article_per_domain.head(5)

### Fetch the articles content for the top domains to check if fetching works


In [ ]:
from langchain_community.document_loaders import NewsURLLoader

TOP_K = 500

print(f"Will fetch one article for each of the top {TOP_K} domains")

urls = one_article_per_domain["url"].tolist()[:TOP_K]

loader = NewsURLLoader(urls, show_progress_bar=True, continue_on_failure=True)

data = []
# data = loader.load()

In [ ]:
for doc in [doc for doc in data if doc.page_content][:10]:
    print(f"Title: {doc.metadata['title']}")
    print(f"URL: {doc.metadata['link']}")
    print(f"Text: {doc.page_content}...")  # Print first 100 characters
    print("---")
    print()

In [ ]:
empty_page_content = [x.metadata["link"] for x in data if not x.page_content]

empty_page_content

In [ ]:
# failed_urls = list(
#     set(urls) - set(x.metadata["link"] for x in data) | set(empty_page_content)
# )
# failed_domains = [get_domain(url) for url in failed_urls]

# failed_domains[:10]



In [ ]:
failed_domains = {
    "skift.com",
    "247wallst.com",
    "windowsreport.com",
    "news-medical.net",
    "heise.de",
    "news18.com",
    "khaleejtimes.com",
    "venturebeat.com",
    "dbta.com",
    "autoevolution.com",
    "finbold.com",
    "techstory.in",
    "ca.finance.yahoo.com",
    "hollywoodreporter.com",
    "pc-tablet.com",
    "cmswire.com",
    "bbc.co.uk",
    "barrons.com",
    "marktechpost.com",
    "bbc.com",
    "businesstoday.in",
    "bgr.com",
    "wfmz.com",
    "geeky-gadgets.com",
    "mobileworldlive.com",
    "latimes.com",
    "philstar.com",
    "9to5mac.com",
    "cio.economictimes.indiatimes.com",
    "dqindia.com",
    "law.com",
    "androidauthority.com",
    "telecoms.com",
    "variety.com",
    "securityboulevard.com",
    "securitysystemsnews.com",
    "uk.pcmag.com",
    "morningstar.com",
    "bworldonline.com",
    "windowscentral.com",
    "djournal.com",
    "marketplace.org",
    "thefastmode.com",
    "baltimoresun.com",
    "crn.com",
    "punchng.com",
    "foxbusiness.com",
    "indiaeducationdiary.in",
    "usnews.com",
    "infoworld.com",
    "theconversation.com",
    "moneycontrol.com",
    "indianexpress.com",
    "microsoft.com",
    "arabianbusiness.com",
    "finanznachrichten.de",
    "news.yahoo.com",
    "livemint.com",
    "yourstory.com",
    "onrec.com",
    "inc42.com",
    "financialit.net",
    "techopedia.com",
    "tech-critter.com",
    "fxstreet.com",
    "computerworld.com",
    "mybroadband.co.za",
    "crowdfundinsider.com",
    "uk.news.yahoo.com",
    "abc.net.au",
    "infoq.com",
    "fiercehealthcare.com",
    "bloomberg.com",
    "benzinga.com",
    "telegraph.co.uk",
    "eagletribune.com",
    "futurism.com",
    "menafn.com",
    "adage.com",
    "semiengineering.com",
    "artificiallawyer.com",
    "wired.com",
    "tweaktown.com",
    "campustechnology.com",
    "sports.yahoo.com",
    "neowin.net",
    "hbr.org",
    "insidebigdata.com",
    "bernama.com",
    "medianama.com",
    "financialpost.com",
    "twinfinite.net",
    "yicaiglobal.com",
    "thehindu.com",
    "rcrwireless.com",
    "computerweekly.com",
    "executivegov.com",
    "businesstimes.com.sg",
    "cnet.com",
    "ciodive.com",
    "es-us.finanzas.yahoo.com",
    "pcmag.com",
    "phonearena.com",
    "pbs.org",
    "gizchina.com",
    "statnews.com",
    "businessghana.com",
    "global.chinadaily.com.cn",
    "businessinsider.in",
    "gizmodo.com",
    "pocket-lint.com",
    "sciencedaily.com",
    "csoonline.com",
    "straitstimes.com",
    "theedgesingapore.com",
    "news.crunchbase.com",
    "knowyourmeme.com",
    "decrypt.co",
    "enterprisetimes.co.uk",
    "news.europawire.eu",
    "kelo.com",
    "pcgamesn.com",
    "manilatimes.net",
    "thederrick.com",
    "itweb.co.za",
    "marketing-interactive.com",
    "nextgov.com",
    "fortune.com",
    "technode.com",
    "techradar.com",
    "livescience.com",
    "edition.cnn.com",
    "theregister.com",
    "tomsguide.com",
    "hackernoon.com",
    "valdostadailytimes.com",
    "ctvnews.ca",
    "businessinsider.com",
    "techspot.com",
    "sg.finance.yahoo.com",
    "thesangaiexpress.com",
    "telegraphindia.com",
    "euronews.com",
    "mirror.co.uk",
    "gulf-times.com",
    "au.pcmag.com",
    "lomeactu.com",
    "economictimes.indiatimes.com",
    "finance.yahoo.com",
    "federalnewsnetwork.com",
    "br.advfn.com",
    "deccanchronicle.com",
    "siliconindia.com",
    "ca.news.yahoo.com",
    "aibusiness.com",
    "scientificamerican.com",
    "ntnews.com.au",
    "eurekalert.org",
    "musicbusinessworldwide.com",
    "businessmirror.com.ph",
    "coingape.com",
    "sharecafe.com.au",
    "economist.com",
    "ign.com",
    "timesunion.com",
    "livebitcoinnews.com",
    "washingtonpost.com",
    "foxnews.com",
    "ippmedia.com",
    "hrdive.com",
    "pressdemocrat.com",
    "rediff.com",
    "beebom.com",
    "thurrott.com",
    "thewindowsclub.com",
    "informazione.it",
    "poynter.org",
    "searchengineland.com",
    "indiatoday.in",
    "eschoolnews.com",
    "digitaljournal.com",
    "stockhouse.com",
    "bleedingcool.com",
    "androguider.com",
    "infosecurity-magazine.com",
    "scmagazine.com",
    "odishatv.in",
    "livewiremarkets.com",
    "securityinfowatch.com",
    "cio.com",
    "pcworld.com",
    "au.finance.yahoo.com",
    "redmondmag.com",
    "cbc.ca",
    "insidebitcoins.com",
    "nbclosangeles.com",
    "fool.com",
    "wsj.com",
    "billboard.com",
    "sg.news.yahoo.com",
    "neurosciencenews.com",
    "sharewise.com",
    "ibtimes.com",
    "forbes.com",
    "nextbigfuture.com",
    "news.webindia123.com",
    "unite.ai",
    "energycentral.com",
    "nzherald.co.nz",
    "newsbytesapp.com",
    "digitaltrends.com",
    "thisdaylive.com",
    "mensjournal.com",
    "beincrypto.com",
    "news.sky.com",
    "breakingdefense.com",
    "tradearabia.com",
    "thenextweb.com",
    "destinationcrm.com",
    "genengnews.com",
    "therobotreport.com",
    "mumbrella.com.au",
    "hoodline.com",
    "techcrunch.com",
    "kahawatungu.com",
    "nasdaq.com",
    "gigazine.net",
    "itnews.com.au",
    "apnews.com",
    "techrepublic.com",
    "sdtimes.com",
    "iblnews.org",
    "investopedia.com",
    "cbsnews.com",
    "technical.ly",
    "seekingalpha.com",
    "inverse.com",
    "geekwire.com",
    "leewayhertz.com",
    "diginomica.com",
    "fmiblog.com",
    "itwire.com",
    "outlookindia.com",
    "cybernews.com",
    "hindustantimes.com",
    "koreaherald.com",
    "aastocks.com",
    "finextra.com",
    "thehackernews.com",
    "techrights.org",
    "techxplore.com",
    "victoriaadvocate.com",
    "slator.com",
    "dallasinnovates.com",
    "saipantribune.com",
    "miamiherald.com",
    "cnn.com",
    "timesofindia.indiatimes.com",
    "komando.com",
    "cyprus-mail.com",
    "travelweekly.com.au",
    "gizmodo.com.au",
    "lablab.ai",
    "datacenterdynamics.com",
    "observer.com",
    "digit.in",
    "firstpost.com",
    "prnewswire.co.uk",
    "devdiscourse.com",
    "animenewsnetwork.com",
    "darkreading.com",
    "arstechnica.com",
    "ft.com",
    "sourcesecurity.com",
    "frontiersin.org",
    "guru3d.com",
    "coinspeaker.com",
    "thewrap.com",
    "telecom.economictimes.indiatimes.com",
    "bangkokpost.com",
    "channelnewsasia.com",
    "news.bloomberglaw.com",
    "technology.inquirer.net",
    "deadline.com",
    "htxt.co.za",
    "joplinglobe.com",
    "thedrum.com",
    "scoop.co.nz",
    "fastcompany.com",
    "technologynetworks.com",
    "newscientist.com",
    "en.globes.co.il",
    "thehill.com",
    "filmibeat.com",
    "time.com",
    "mobilemarketingmagazine.com",
    "medscape.com",
    "infotechlead.com",
    "biometricupdate.com",
    "theatlantic.com",
    "thehindubusinessline.com",
    "hackaday.com",
    "theprint.in",
    "eurogamer.net",
    "businesswire.com",
    "money.rediff.com",
    "indiatvnews.com",
    "cacm.acm.org",
    "ndtv.com",
    "it-online.co.za",
    "news.marketersmedia.com",
    "asia.nikkei.com",
    "tvbeurope.com",
    "govtech.com",
    "jpost.com",
    "daytondailynews.com",
    "vietbao.vn",
    "linkedin.com",
    "technologyreview.com",
    "dailytelegraph.com.au",
    "cryptobriefing.com",
    "advanced-television.com",
    "pocketgamer.com",
    "cryptopolitan.com",
    "gamespot.com",
    "ibtimes.co.uk",
    "visualstudiomagazine.com",
    "cnbctv18.com",
    "vir.com.vn",
    "nypost.com",
    "entrepreneur.com",
    "freepressjournal.in",
    "thenews-chronicle.com",
    "theglobeandmail.com",
    "gizbot.com",
    "news.microsoft.com",
    "singularityhub.com",
    "networkworld.com",
    "techbullion.com",
    "thestar.com.my",
    "electronicdesign.com",
    "cambridge.org",
    "ocbj.com",
    "beckershospitalreview.com",
    "gadgets360.com",
    "smh.com.au",
    "popsci.com",
    "electronics360.globalspec.com",
    "nbcnews.com",
    "markets.businessinsider.com",
    "nytimes.com",
    "techzine.eu",
    "albawaba.com",
    "asiaone.com",
    "financialexpress.com",
    "chicagotribune.com",
    "thejournal.com",
    "theinformation.com",
    "campaignasia.com",
    "uk.investing.com",
    "siliconrepublic.com",
    "insidehpc.com",
    "designnews.com",
    "daijiworld.com",
    "au.news.yahoo.com",
    "insidermonkey.com",
    "in.mashable.com",
    "thehansindia.com",
    "chinadaily.com.cn",
    "fudzilla.com",
    "scitechdaily.com",
    "bignewsnetwork.com",
    "irishtimes.com",
    "brandequity.economictimes.indiatimes.com",
    "phys.org",
    "tradersmagazine.com",
    "goshennews.com",
    "analyticsindiamag.com",
    "appleinsider.com",
    "gazette.com",
    "hcamag.com",
    "adexchanger.com",
    "aboutamazon.com",
    "thesun.co.uk",
    "9to5google.com",
    "digitimes.com",
    "news.com.au",
    "business-standard.com",
    "nbcnewyork.com",
    "uk.finance.yahoo.com",
    "fbcnews.com.fj",
    "edweek.org",
    "crypto-news-flash.com",
    "deccanherald.com",
    "tmcnet.com",
    "china.org.cn",
    "politico.com",
    "newswit.com",
    "gulfnews.com",
    "dexerto.com",
    "vox.com",
    "dailydot.com",
    "lifehacker.com",
    "bizjournals.com",
    "lelezard.com",
    "dailymail.co.uk",
    "androidpolice.com",
    "medindia.net",
    "dailyhodl.com",
    "cryptonews.com",
    "dcvelocity.com",
    "searchenginejournal.com",
    "buffalo.edu",
    "macrumors.com",
    "cointelegraph.com",
    "inc.com",
    "jdsupra.com",
    "gamerant.com",
    "mmm-online.com",
    "fiverr.com",
    "newsweek.com",
    "pionline.com",
    "syncedreview.com",
    "mg.co.za",
    "psychologytoday.com",
    "crn.com.au",
    "stuff.tv",
    "couriermail.com.au",
    "tribune.com.pk",
    "scmp.com",
    "globaltimes.cn",
    "androidheadlines.com",
    "nocamels.com",
    "playtoearngames.com",
    "theverge.com",
    "english.aawsat.com",
    "malaysiasun.com",
    "techreport.com",
    "newatlas.com",
    "hothardware.com",
    "bleepingcomputer.com",
    "tbsnews.net",
    "sdxcentral.com",
    "investing.com",
    "independent.co.uk",
    "bbntimes.com",
    "msn.com",
    "tribuneindia.com",
    "abcnews.go.com",
    "securityweek.com",
    "newindianexpress.com",
    "design-reuse.com",
    "techtimes.com",
    "marketwatch.com",
    "bandt.com.au",
    "taiwannews.com.tw",
    "gizmochina.com",
    "rappler.com",
    "channelnews.com.au",
    "zawya.com",
    "washingtontimes.com",
    "fonearena.com",
    "fedscoop.com",
    "dailymemphian.com",
    "news.stocktradersdaily.com",
    "betakit.com",
    "yahoo.com",
    "themalaysianreserve.com",
    "zdnet.com",
    "afr.com",
    "israel21c.org",
    "siliconangle.com",
    "datanami.com",
    "seattletimes.com",
    "digitalinformationworld.com",
    "koreatimes.co.kr",
    "standard.co.uk",
    "sfgate.com",
    "tech.hindustantimes.com",
    "accountingtoday.com",
    "americanbanker.com",
    "econotimes.com",
    "bostonglobe.com",
    "01net.it",
    "money.usnews.com",
    "nature.com",
    "thestreet.com",
    "campaignlive.co.uk",
    "medicalxpress.com",
    "zeebiz.com",
    "mashable.com",
    "taipeitimes.com",
    "usatoday.com",
    "manilastandard.net",
    "defenseone.com",
    "insidehighered.com",
    "wgnradio.com",
    "smartcompany.com.au",
    "theaustralian.com.au",
    "reuters.com",
}

In [ ]:
for doc in [doc for doc in data if doc.page_content][:2]:
    print(f"Title: {doc.metadata['title']}")
    print(f"URL: {doc.metadata['link']}")
    print(f"Text: {doc.page_content}...")  # Print first 100 characters
    print("---")
    print()

# Fetch the missing articles using `newspaper3k`


In [ ]:
import time
import random
from newspaper import Article
from tqdm import tqdm
import pandas as pd
from collections import defaultdict


def process_articles(
    df, limit: int | None = None, failed_domains: set[str] | None = None, max_failures=3
):
    """
    Process articles from a DataFrame, fetching content for each article URL.

    This function iterates through the provided DataFrame, attempting to fetch
    the content for each article URL. It respects rate limiting, skips articles
    from failed domains, stops processing after reaching the specified limit,
    and tracks new domains that consistently fail.

    Parameters:
    -----------
    df : pandas.DataFrame
        A DataFrame containing article information. Must include 'url' and
        'page_content' columns.
    limit : int, optional
        The maximum number of articles to process. If None, all articles are processed.
    failed_domains : set, optional
        A set of domain names to skip (e.g., {'example.com', 'faildomain.com'}).
    max_failures : int, optional
        The number of failures allowed for a domain before it's added to ignored_domains.

    Returns:
    --------
    tuple
        A tuple containing:
        - The updated DataFrame with fetched content in the 'page_content' column.
        - A set of newly ignored domains.

    Side Effects:
    -------------
    - Updates the input DataFrame in-place.
    """
    failed_domains = failed_domains or set()
    new_failed_domains = set()
    domain_failures = defaultdict(int)

    previous_domain = None
    updated_count = 0
    skipped_count = 0
    processed_count = 0

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing articles"):
        if limit is not None and processed_count >= limit:
            print(f"Reached the limit of {limit} articles. Stopping.")
            break

        url = row["url"]
        current_domain = get_domain(url)

        # Skip if the domain is in the failed_domains or ignored_domains list
        if current_domain in failed_domains or current_domain in new_failed_domains:
            # print(f"Skipping {url} (domain failing)")
            skipped_count += 1
            continue

        # Skip if content already exists
        if pd.notna(row["page_content"]):
            # print(f"Content already exists for {url}")
            skipped_count += 1
            continue

        # If the domain is the same as the previous request, add a delay
        if current_domain == previous_domain:
            time.sleep(random.uniform(1, 3))

        content = fetch_one_article(url)

        if content:
            # Update the DataFrame
            df.at[_, "page_content"] = content
            print(f"Updated content for {url}")
            updated_count += 1
            # Reset failure count for successful fetch
            domain_failures[current_domain] = 0
        else:
            print(f"No content fetched for {url}")
            skipped_count += 1
            # Increment failure count for the domain
            domain_failures[current_domain] += 1

            # Check if domain should be ignored
            if domain_failures[current_domain] >= max_failures:
                new_failed_domains.add(current_domain)
                print(
                    f"Added {current_domain} to ignored domains after {max_failures} failures"
                )

        previous_domain = current_domain
        processed_count += 1

    print(f"Updated {updated_count} articles, skipped {skipped_count} articles")
    print(f"Total processed: {processed_count}")
    print(f"Newly ignored domains: {new_failed_domains}")
    return df, new_failed_domains


# Example usage
try:
    df, new_ignored_domains = process_articles(
        df, limit=None, failed_domains=set(failed_domains)
    )

    # Update failed_domains with newly ignored domains
    failed_domains.update(new_ignored_domains)

except KeyboardInterrupt:
    print("Process interrupted by user. Progress saved.")

### Export updated df


In [ ]:
from datetime import datetime
from os.path import join

output = join("data", f"blogdb.ai_news_{datetime.now().strftime('%Y_%m_%d_%H_%M')}.csv")

df.to_csv(output, index=False, encoding="utf-8")

# Update mongoDB with the page_content
